# DARP ML training data — exploration

Data-only pass on the cached labeled-state dataset (`ml_train/cache_v2_dynamic/`).
This notebook does **not** load any trained models — that's deferred until after
the next retrain on the combined v1 + v2 log set.

The four questions this notebook answers:

1. How are the binary labels distributed across the 20 operator pairs?
2. What does the underlying continuous `improvement_freq` look like before binarization at threshold 0.3?
3. **At each state, how many of the 20 pairs are simultaneously labeled positive?** (Are most states ambiguous or selective?)
4. What do the 20 features look like distributionally, and which are redundant via correlation?


## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure helpers.py is importable regardless of where the kernel is launched.
_HERE = Path.cwd()
if (_HERE / "helpers.py").exists():
    sys.path.insert(0, str(_HERE))
elif (_HERE / "analysis" / "helpers.py").exists():
    sys.path.insert(0, str(_HERE / "analysis"))

from helpers import (
    BATCH_COLORS, DESTROY_COLORS, DESTROY_OPS, REPAIR_OPS,
    IMPROVEMENT_FREQ_THRESHOLD,
    feature_batch_map, feature_columns,
    load_features_labels, pair_freq_columns, pair_label_columns,
    pair_label_to_name, parse_tier, save_fig,
)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.bbox"] = "tight"


## 1. Dataset overview

In [ ]:
features_df, labels_df = load_features_labels()

print(f"features.parquet : {features_df.shape[0]:>6d} rows x {features_df.shape[1]:>2d} cols")
print(f"labels.parquet   : {labels_df.shape[0]:>6d} rows x {labels_df.shape[1]:>2d} cols")
print(f"unique log files : {labels_df['log_file'].nunique()}")
print()

# Per-tier state count.
labels_df["tier"] = labels_df["log_file"].apply(parse_tier)
features_df["tier"] = features_df["log_file"].apply(parse_tier)
tier_counts = labels_df["tier"].value_counts().reindex(
    ["30s", "01min", "02min", "05min", "15min"], fill_value=0,
)
print("States per tier (in labels.parquet):")
print(tier_counts.to_string())
print()
print(f"Total labeled states     : {len(labels_df):>6d}")
print(f"Total feature rows       : {len(features_df):>6d}")
print(f"Note: labels.parquet may be smaller if epoch-1 / uninformative states were filtered.")


## 2. Per-pair binary label distribution

For each of the 20 operator pairs, what fraction of states have the pair labeled positive (= the pair improves often enough to clear `improvement_freq >= 0.3`)?

Pairs with positive rate <5% or >95% are **degenerate** — the classifier can't learn meaningful signal from them and they're excluded from the median-F1-lift summary in training reports.


In [ ]:
pair_cols = pair_label_columns(labels_df)
pos_rates = labels_df[pair_cols].mean()  # binary mean = positive rate

# Build display table.
rows = []
for c, rate in pos_rates.items():
    d, r = pair_label_to_name(c)
    if rate < 0.05:
        status = "DEGEN low"
    elif rate > 0.95:
        status = "DEGEN high"
    else:
        status = "balanced"
    rows.append({
        "pair": f"{d}+{r}",
        "n_positive": int(labels_df[c].sum()),
        "n_total": len(labels_df),
        "positive_rate": rate,
        "status": status,
    })
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print()
n_degen = (summary_df["status"] != "balanced").sum()
print(f"Degenerate pairs: {n_degen}/20  (excluded from median-lift in training reports)")


In [ ]:
# Plot per-pair positive rate as bars, color-grouped by destroy operator.
fig, ax = plt.subplots(figsize=(11, 4.5))
labels = []
heights = []
colors = []
for c in pair_cols:
    d, r = pair_label_to_name(c)
    labels.append(f"{d}+{r}")
    heights.append(pos_rates[c])
    colors.append(DESTROY_COLORS[d])

x = np.arange(len(labels))
ax.bar(x, heights, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(0.05, color="grey", lw=0.8, ls=":", label="degeneracy bands (5%, 95%)")
ax.axhline(0.95, color="grey", lw=0.8, ls=":")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Positive label rate")
ax.set_title("Per-pair positive label rate (label = improvement_freq >= 0.3)")
ax.set_ylim(0, 1.0)
ax.legend(loc="upper right", fontsize=8)

# Color legend for destroy ops.
from matplotlib.patches import Patch
destroy_legend = [Patch(facecolor=DESTROY_COLORS[d], label=d) for d in DESTROY_OPS]
ax.legend(handles=destroy_legend, loc="upper left", fontsize=8, title="destroy op")

plt.tight_layout()
save_fig(fig, "02_per_pair_positive_rate")
plt.show()


## 3. Continuous `improvement_freq` distribution per pair

`labels.parquet` carries both the binary `pair_<d>_<r>` columns AND the underlying
continuous `improve_freq_<d>_<r>` columns. The binary label is just `improve_freq >= 0.3`.

This grid shows the full distribution of the continuous value per pair so you can
see how labels would shift under a different threshold (e.g. 0.2 or 0.4) — useful for the methodology chapter's threshold-sensitivity discussion.

Red vertical line = current threshold (0.3).


In [ ]:
freq_cols = pair_freq_columns(labels_df)

fig, axes = plt.subplots(5, 4, figsize=(13, 14), sharex=True, sharey=False)
for di, d in enumerate(DESTROY_OPS):
    for ri, r in enumerate(REPAIR_OPS):
        col = f"improve_freq_{di}_{ri}"
        ax = axes[di, ri]
        if col not in labels_df.columns:
            ax.set_visible(False)
            continue
        vals = labels_df[col].values
        ax.hist(vals, bins=21, range=(0, 1.0),
                color=DESTROY_COLORS[d], edgecolor="black", linewidth=0.4)
        ax.axvline(IMPROVEMENT_FREQ_THRESHOLD, color="red", lw=1.2,
                   label=f"thresh={IMPROVEMENT_FREQ_THRESHOLD}")
        pos_rate = (vals >= IMPROVEMENT_FREQ_THRESHOLD).mean()
        ax.set_title(f"{d}+{r}\npos_rate = {pos_rate:.2%}", fontsize=9)
        ax.set_xlim(0, 1.0)
        if di == 4:
            ax.set_xlabel("improvement_freq")
        if ri == 0:
            ax.set_ylabel("# states")

fig.suptitle("Distribution of improve_freq per pair  (binarization threshold = 0.3, red line)",
             fontsize=12, y=1.00)
plt.tight_layout()
save_fig(fig, "03_improvement_freq_grid")
plt.show()


## 4. State-level improvement-options distribution

**The key question**: for each state, how many of the 20 pairs are simultaneously
labeled positive?

- If most states have **0-2 positive pairs**, the data is highly *selective* — only one or two operator-pair choices would have improved the cost from this state. The model has a clear target.
- If most states have **15+ positive pairs**, the data is *ambiguous* — almost any pair would have improved. The model has to learn nuanced ordering rather than a clear classification.
- If the distribution is **bimodal**, you may have qualitatively different state regimes.

This is one of the most informative views into "what is the labeling signal actually telling us".


In [ ]:
# Per-state count: how many of the 20 pair_<d>_<r> columns are 1?
pair_cols = pair_label_columns(labels_df)
labels_df["improvement_count"] = labels_df[pair_cols].sum(axis=1)

# Print summary stats.
ic = labels_df["improvement_count"]
print(f"States analyzed: {len(ic):,}")
print()
print(f"  mean   {ic.mean():.2f}")
print(f"  median {ic.median():.0f}")
print(f"  mode   {int(ic.mode().iloc[0])}")
print(f"  std    {ic.std():.2f}")
print(f"  min    {ic.min():.0f}")
print(f"  max    {ic.max():.0f}")
print()
print("Percentile breakdown (count of positive pairs per state):")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>2d} = {int(np.percentile(ic, p)):>2d}")


In [ ]:
# Histogram + cumulative.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram, bars at integer values 0..20.
ax1.hist(ic, bins=np.arange(-0.5, 21.5, 1.0),
         color="#4c72b0", edgecolor="black", linewidth=0.5)
ax1.set_xlabel("# pairs labeled positive at this state (out of 20)")
ax1.set_ylabel("# states")
ax1.set_title("State-level improvement-options distribution")
ax1.axvline(ic.mean(), color="red", ls="--", lw=1.2, label=f"mean={ic.mean():.1f}")
ax1.axvline(ic.median(), color="orange", ls="--", lw=1.2, label=f"median={ic.median():.0f}")
ax1.legend(fontsize=9)
ax1.set_xticks(range(0, 21))

# Right: per-tier breakdown — does ambiguity scale with instance size?
tier_order = ["30s", "01min", "02min", "05min", "15min"]
present_tiers = [t for t in tier_order if (labels_df["tier"] == t).any()]
for t in present_tiers:
    sub = labels_df.loc[labels_df["tier"] == t, "improvement_count"]
    if len(sub) == 0:
        continue
    ax2.hist(sub, bins=np.arange(-0.5, 21.5, 1.0),
             alpha=0.4, label=f"{t}  (n={len(sub):,})", density=True)
ax2.set_xlabel("# pairs labeled positive")
ax2.set_ylabel("density")
ax2.set_title("Stratified by instance tier (density-normalized)")
ax2.legend(fontsize=8)
ax2.set_xticks(range(0, 21, 2))

plt.tight_layout()
save_fig(fig, "04_improvement_options_distribution")
plt.show()


In [ ]:
# Discrete breakdown: how often does each exact count occur?
print("Distribution of improvement_count (exact counts):")
print()
counts = ic.value_counts().sort_index()
total = len(ic)
print(f"  count  states  pct")
for k, v in counts.items():
    bar = "*" * int(60 * v / total)
    print(f"   {int(k):>3d}   {v:>6d}  {100*v/total:>5.2f}%  {bar}")


## 5. Feature distributions and correlation

The 20 features in the registry, color-grouped by batch. Each subplot is a histogram
of one feature across all states. The correlation heatmap (lower) shows Pearson r
between every pair of features — strong correlations (|r| > 0.8) hint at redundant
features that could potentially be collapsed.


In [ ]:
batch_map = feature_batch_map()
# Source-of-truth: the registry. Only analyze columns that are real features.
# (features_df may have a `tier` column we added in section 1, plus
# log_file/iter join keys.)
feat_cols = [c for c in features_df.columns if c in batch_map]

# Order features by batch so the grid groups them visually.
batch_order = ["core", "tightness", "worst_support", "lns_state", "rolling_horizon"]
def _sort_key(c):
    b = batch_map[c]
    rank = batch_order.index(b) if b in batch_order else len(batch_order)
    return (rank, c)
feat_cols_sorted = sorted(feat_cols, key=_sort_key)

n = len(feat_cols_sorted)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.6 * nrows))
axes = axes.flatten()
for i, c in enumerate(feat_cols_sorted):
    ax = axes[i]
    batch = batch_map.get(c, "?")
    color = BATCH_COLORS.get(batch, "grey")
    vals = features_df[c].values
    ax.hist(vals, bins=40, color=color, edgecolor="black", linewidth=0.3)
    ax.set_title(f"{c}\n(batch: {batch})", fontsize=8)
    ax.tick_params(labelsize=7)
for j in range(n, len(axes)):
    axes[j].set_visible(False)

# Build a batch-color legend.
from matplotlib.patches import Patch
legend = [Patch(facecolor=BATCH_COLORS[b], label=b) for b in batch_order]
fig.legend(handles=legend, loc="upper center", ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, 1.02))

fig.suptitle("Feature distributions (color = batch)", fontsize=12, y=1.05)
plt.tight_layout()
save_fig(fig, "05a_feature_distributions")
plt.show()


In [ ]:
# Pearson correlation heatmap, batch-ordered features.
corr = features_df[feat_cols_sorted].corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
ax.set_xticks(range(len(feat_cols_sorted)))
ax.set_yticks(range(len(feat_cols_sorted)))
ax.set_xticklabels(feat_cols_sorted, rotation=70, ha="right", fontsize=7)
ax.set_yticklabels(feat_cols_sorted, fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label="Pearson r")
ax.set_title("Feature correlation matrix (batch-ordered)")

# Draw lines separating batches.
batch_boundaries = []
last_b = None
for i, c in enumerate(feat_cols_sorted):
    b = batch_map.get(c, "?")
    if b != last_b and last_b is not None:
        batch_boundaries.append(i - 0.5)
    last_b = b
for x in batch_boundaries:
    ax.axhline(x, color="black", lw=0.7)
    ax.axvline(x, color="black", lw=0.7)

plt.tight_layout()
save_fig(fig, "05b_feature_correlation_heatmap")
plt.show()


In [ ]:
# Strongly-correlated pairs (|r| > 0.8) — candidates for redundancy.
strong = []
for i, a in enumerate(feat_cols_sorted):
    for j, b in enumerate(feat_cols_sorted):
        if j <= i:
            continue
        r = corr.loc[a, b]
        if abs(r) > 0.8:
            strong.append((a, b, r))

if not strong:
    print("No feature pairs with |r| > 0.8 — all features carry distinct signal.")
else:
    print(f"Strongly-correlated feature pairs (|r| > 0.8):  {len(strong)}")
    print()
    print(f"{'feature_a':<32s} {'feature_b':<32s}  r")
    print("-" * 80)
    for a, b, r in sorted(strong, key=lambda x: -abs(x[2])):
        print(f"{a:<32s} {b:<32s}  {r:+.3f}")


## Notes for the next pass

This notebook deliberately does **not** load any trained models. After the next
retrain on the combined v1 + v2 log set, a sibling notebook (planned name:
`model_performance.ipynb`) will cover:

- Per-classifier F1 lift / precision / recall / confusion matrices
- Per-classifier feature importance heatmap (one row per feature, one column per pair)
- Cross-model comparisons (5f vs 13f vs 20f) per pair
- Identification of pairs that benefit most from feature additions

All figures from this notebook are saved as PNGs in `analysis/figures/` for inclusion in writeups.
